# Titanic Dataset: Imputation Strategies and Model Comparison

This section will demonstrate two different imputation strategies for categorical features on the Titanic dataset: Frequent-Value Imputation and Missing-Category Imputation. We will then train a simple classification model (Logistic Regression) on each preprocessed dataset and compare their accuracies.

## 1. Load and Initial Exploration of the Titanic Dataset

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# Load the Titanic dataset
# Using a direct URL for reliability, as 'sample_data' might not always contain it.
try:
    df_titanic = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
    print("Titanic dataset loaded successfully.")
except Exception as e:
    print(f"Error loading dataset: {e}. Trying local file if available...")
    df_titanic = pd.read_csv('/content/sample_data/titanic_train.csv') # Fallback to local


print("\nFirst 5 rows of the dataset:")
display(df_titanic.head())

print("\nDataset Information:")
df_titanic.info()

print("\nMissing values count per column:")
display(df_titanic.isnull().sum().sort_values(ascending=False))

Titanic dataset loaded successfully.

First 5 rows of the dataset:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Missing values count per column:


,0
Cabin,687
Age,177
Embarked,2
PassengerId,0
Name,0
Pclass,0
Survived,0
Sex,0
Parch,0
SibSp,0


## 2. Preprocessing for Imputation and Modeling

We'll prepare common steps for both imputation strategies, including handling numerical missing values (using median) and encoding categorical features after imputation.

In [8]:
# Drop columns that are unlikely to be useful or have too many missing values initially
df_titanic = df_titanic.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# Separate features (X) and target (y)
X = df_titanic.drop('Survived', axis=1)
y = df_titanic['Survived']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

Shape of X_train: (712, 7)
Shape of X_test: (179, 7)


## 3. Imputation Strategy 1: Frequent-Value Imputation for Categorical Features

For this strategy, we will replace missing categorical values with the mode (most frequent value) of their respective columns. Numerical missing values will be imputed with the median.

In [9]:
# Create copies for this strategy
X_train_freq = X_train.copy()
X_test_freq = X_test.copy()

# Identify numerical and categorical columns with missing values in the training set
numerical_cols_with_missing = X_train_freq.select_dtypes(include=np.number).columns[X_train_freq.select_dtypes(include=np.number).isnull().any()].tolist()
categorical_cols_with_missing = X_train_freq.select_dtypes(exclude=np.number).columns[X_train_freq.select_dtypes(exclude=np.number).isnull().any()].tolist()

print("Numerical columns with missing values:", numerical_cols_with_missing)
print("Categorical columns with missing values:", categorical_cols_with_missing)

# 3.1. Impute numerical columns with Median
for col in numerical_cols_with_missing:
    median_val = X_train_freq[col].median()
    X_train_freq[col].fillna(median_val, inplace=True)
    X_test_freq[col].fillna(median_val, inplace=True)
    print(f"Numerical column '{col}' imputed with median: {median_val:.2f}")

# 3.2. Impute categorical columns with Mode (Frequent-Value)
for col in categorical_cols_with_missing:
    mode_val = X_train_freq[col].mode()[0] # get the first mode if there are multiple
    X_train_freq[col].fillna(mode_val, inplace=True)
    X_test_freq[col].fillna(mode_val, inplace=True)
    print(f"Categorical column '{col}' imputed with mode: '{mode_val}'")

print("\nMissing values after frequent-value imputation (training set):\n", X_train_freq.isnull().sum().sum())
print("Missing values after frequent-value imputation (test set):\n", X_test_freq.isnull().sum().sum())

# 3.3. One-Hot Encode categorical features
# Identify all categorical columns after imputation
all_categorical_cols = X_train_freq.select_dtypes(include=['object']).columns
X_train_freq_encoded = pd.get_dummies(X_train_freq, columns=all_categorical_cols, drop_first=True)
X_test_freq_encoded = pd.get_dummies(X_test_freq, columns=all_categorical_cols, drop_first=True)

# Align columns between train and test (important for one-hot encoding)
X_train_freq_encoded, X_test_freq_encoded = X_train_freq_encoded.align(X_test_freq_encoded, join='left', axis=1, fill_value=0)

print("\nShape after one-hot encoding (training set):", X_train_freq_encoded.shape)

# 3.4. Train and Evaluate Logistic Regression Model (Frequent-Value)
model_freq = LogisticRegression(random_state=42, solver='liblinear', max_iter=200)
model_freq.fit(X_train_freq_encoded, y_train)

y_pred_freq = model_freq.predict(X_test_freq_encoded)
accuracy_freq = accuracy_score(y_test, y_pred_freq)
print(f"\nAccuracy with Frequent-Value Imputation: {accuracy_freq:.4f}")

Numerical columns with missing values: ['Age']
Categorical columns with missing values: ['Embarked']
Numerical column 'Age' imputed with median: 28.00
Categorical column 'Embarked' imputed with mode: 'S'

Missing values after frequent-value imputation (training set):
 0
Missing values after frequent-value imputation (test set):
 0

Shape after one-hot encoding (training set): (712, 8)

Accuracy with Frequent-Value Imputation: 0.7821


/tmp/ipykernel_13744/3701611440.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train_freq[col].fillna(median_val, inplace=True)
/tmp/ipykernel_13744/3701611440.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

## 4. Imputation Strategy 2: Missing-Category Imputation for Categorical Features

For this strategy, we will replace missing categorical values with a new category called 'Missing'. Numerical missing values will still be imputed with the median.

In [10]:
# Create fresh copies for this strategy
X_train_missing = X_train.copy()
X_test_missing = X_test.copy()

# Identify numerical and categorical columns with missing values in the training set
numerical_cols_with_missing = X_train_missing.select_dtypes(include=np.number).columns[X_train_missing.select_dtypes(include=np.number).isnull().any()].tolist()
categorical_cols_with_missing = X_train_missing.select_dtypes(exclude=np.number).columns[X_train_missing.select_dtypes(exclude=np.number).isnull().any()].tolist()

print("Numerical columns with missing values:", numerical_cols_with_missing)
print("Categorical columns with missing values:", categorical_cols_with_missing)

# 4.1. Impute numerical columns with Median
for col in numerical_cols_with_missing:
    median_val = X_train_missing[col].median()
    X_train_missing[col].fillna(median_val, inplace=True)
    X_test_missing[col].fillna(median_val, inplace=True)
    print(f"Numerical column '{col}' imputed with median: {median_val:.2f}")

# 4.2. Impute categorical columns by adding 'Missing' category
for col in categorical_cols_with_missing:
    X_train_missing[col].fillna('Missing', inplace=True)
    X_test_missing[col].fillna('Missing', inplace=True)
    print(f"Categorical column '{col}' imputed with 'Missing' category.")

print("\nMissing values after missing-category imputation (training set):\n", X_train_missing.isnull().sum().sum())
print("Missing values after missing-category imputation (test set):\n", X_test_missing.isnull().sum().sum())

# 4.3. One-Hot Encode categorical features
# Identify all categorical columns after imputation
all_categorical_cols_missing = X_train_missing.select_dtypes(include=['object']).columns
X_train_missing_encoded = pd.get_dummies(X_train_missing, columns=all_categorical_cols_missing, drop_first=True)
X_test_missing_encoded = pd.get_dummies(X_test_missing, columns=all_categorical_cols_missing, drop_first=True)

# Align columns between train and test
X_train_missing_encoded, X_test_missing_encoded = X_train_missing_encoded.align(X_test_missing_encoded, join='left', axis=1, fill_value=0)

print("\nShape after one-hot encoding (training set):", X_train_missing_encoded.shape)

# 4.4. Train and Evaluate Logistic Regression Model (Missing-Category)
model_missing = LogisticRegression(random_state=42, solver='liblinear', max_iter=200)
model_missing.fit(X_train_missing_encoded, y_train)

y_pred_missing = model_missing.predict(X_test_missing_encoded)
accuracy_missing = accuracy_score(y_test, y_pred_missing)
print(f"\nAccuracy with Missing-Category Imputation: {accuracy_missing:.4f}")

Numerical columns with missing values: ['Age']
Categorical columns with missing values: ['Embarked']
Numerical column 'Age' imputed with median: 28.00
Categorical column 'Embarked' imputed with 'Missing' category.

Missing values after missing-category imputation (training set):
 0
Missing values after missing-category imputation (test set):
 0

Shape after one-hot encoding (training set): (712, 9)

Accuracy with Missing-Category Imputation: 0.7821


/tmp/ipykernel_13744/2802109707.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train_missing[col].fillna(median_val, inplace=True)
/tmp/ipykernel_13744/2802109707.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

## 5. Comparison of Accuracies

When choosing an imputation strategy, it's crucial to understand not just the immediate impact on model accuracy, but also how each method might affect feature distributions, relationships between variables, and the interpretability of the model. Even if two methods yield similar overall accuracy, their effects on the underlying data can differ.

-   **Frequent-Value Imputation (Mode Imputation)**: This method replaces missing values with the most common category. It's simple and can be effective when the missingness is random and the missing proportion is small. However, it can distort the distribution of the categorical variable by artificially inflating the frequency of the most common category, potentially leading to biased estimates if the missingness mechanism is not truly random.

-   **Missing-Category Imputation**: This method treats 'missing' as a distinct category. This approach can be very informative, especially when the fact that a value is missing is itself a significant piece of information (e.g., a missing 'Cabin' might imply a lower socio-economic status). It avoids altering the original distribution of observed categories and allows the model to learn a specific weight for the missingness.

Let's visualize the accuracies to clearly see the comparison.

In [11]:
print(f"Accuracy with Frequent-Value Imputation: {accuracy_freq:.4f}")
print(f"Accuracy with Missing-Category Imputation: {accuracy_missing:.4f}")

if accuracy_freq > accuracy_missing:
    print("\nFrequent-Value Imputation resulted in slightly higher accuracy for this model and dataset.")
elif accuracy_missing > accuracy_freq:
    print("\nMissing-Category Imputation resulted in slightly higher accuracy for this model and dataset.")
else:
    print("\nBoth imputation strategies yielded similar accuracy for this model and dataset.")

Accuracy with Frequent-Value Imputation: 0.7821
Accuracy with Missing-Category Imputation: 0.7821

Both imputation strategies yielded similar accuracy for this model and dataset.


# Data Imputation Demo

First, let's create a sample DataFrame with some missing values in both numerical and categorical columns.

In [1]:
import pandas as pd
import numpy as np

# Create a sample DataFrame with missing values
data = {
    'Numerical_Col_A': [10, 20, np.nan, 40, 50, 60, np.nan, 80, 90, 100],
    'Numerical_Col_B': [1, np.nan, 3, 4, 5, np.nan, 7, 8, 9, 10],
    'Categorical_Col_C': ['A', 'B', 'A', np.nan, 'C', 'B', 'A', 'C', np.nan, 'A'],
    'Categorical_Col_D': ['X', 'Y', 'Z', 'X', np.nan, 'Y', 'Z', 'X', 'Y', np.nan]
}
df = pd.DataFrame(data)

print("Original DataFrame with missing values:")
display(df)

Original DataFrame with missing values:


,Numerical_Col_A,Numerical_Col_B,Categorical_Col_C,Categorical_Col_D
0,10.0,1.0,A,X
1,20.0,NaN,B,Y
2,NaN,3.0,A,Z
3,40.0,4.0,NaN,X
4,50.0,5.0,C,NaN
5,60.0,NaN,B,Y
6,NaN,7.0,A,Z
7,80.0,8.0,C,X
8,90.0,9.0,NaN,Y
9,100.0,10.0,A,NaN


Now, let's identify numerical and categorical columns to apply the correct imputation strategy.

In [2]:
# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

print("Numerical Columns:", numerical_cols.tolist())
print("Categorical Columns:", categorical_cols.tolist())

Numerical Columns: ['Numerical_Col_A', 'Numerical_Col_B']
Categorical Columns: ['Categorical_Col_C', 'Categorical_Col_D']


### Mean Imputation for Numerical Columns

We will fill missing values in numerical columns with their respective means. This is suitable for normally distributed data without significant outliers.

In [3]:
df_mean_imputed = df.copy()
for col in numerical_cols:
    if df_mean_imputed[col].isnull().any():
        mean_value = df_mean_imputed[col].mean()
        df_mean_imputed[col].fillna(mean_value, inplace=True)
        print(f"Missing values in '{col}' filled with mean: {mean_value:.2f}")

print("\nDataFrame after Mean Imputation for numerical columns:")
display(df_mean_imputed)

Missing values in 'Numerical_Col_A' filled with mean: 56.25
Missing values in 'Numerical_Col_B' filled with mean: 5.88

DataFrame after Mean Imputation for numerical columns:


/tmp/ipykernel_13744/3692255597.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_mean_imputed[col].fillna(mean_value, inplace=True)


,Numerical_Col_A,Numerical_Col_B,Categorical_Col_C,Categorical_Col_D
0,10.00,1.000,A,X
1,20.00,5.875,B,Y
2,56.25,3.000,A,Z
3,40.00,4.000,NaN,X
4,50.00,5.000,C,NaN
5,60.00,5.875,B,Y
6,56.25,7.000,A,Z
7,80.00,8.000,C,X
8,90.00,9.000,NaN,Y
9,100.00,10.000,A,NaN


### Median Imputation for Numerical Columns

Alternatively, we can fill missing values in numerical columns with their medians. This is often preferred when data has outliers or is skewed, as the median is more robust to extreme values.

In [4]:
df_median_imputed = df.copy()
for col in numerical_cols:
    if df_median_imputed[col].isnull().any():
        median_value = df_median_imputed[col].median()
        df_median_imputed[col].fillna(median_value, inplace=True)
        print(f"Missing values in '{col}' filled with median: {median_value:.2f}")

print("\nDataFrame after Median Imputation for numerical columns:")
display(df_median_imputed)

Missing values in 'Numerical_Col_A' filled with median: 55.00
Missing values in 'Numerical_Col_B' filled with median: 6.00

DataFrame after Median Imputation for numerical columns:


/tmp/ipykernel_13744/703929340.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_median_imputed[col].fillna(median_value, inplace=True)


,Numerical_Col_A,Numerical_Col_B,Categorical_Col_C,Categorical_Col_D
0,10.0,1.0,A,X
1,20.0,6.0,B,Y
2,55.0,3.0,A,Z
3,40.0,4.0,NaN,X
4,50.0,5.0,C,NaN
5,60.0,6.0,B,Y
6,55.0,7.0,A,Z
7,80.0,8.0,C,X
8,90.0,9.0,NaN,Y
9,100.0,10.0,A,NaN


### Mode Imputation for Categorical Columns

For categorical columns, we will fill missing values with the mode (most frequent value).

In [5]:
df_mode_imputed = df.copy()
for col in categorical_cols:
    if df_mode_imputed[col].isnull().any():
        mode_value = df_mode_imputed[col].mode()[0] # .mode() returns a Series, take the first if multiple modes
        df_mode_imputed[col].fillna(mode_value, inplace=True)
        print(f"Missing values in '{col}' filled with mode: {mode_value}")

print("\nDataFrame after Mode Imputation for categorical columns:")
display(df_mode_imputed)

Missing values in 'Categorical_Col_C' filled with mode: A
Missing values in 'Categorical_Col_D' filled with mode: X

DataFrame after Mode Imputation for categorical columns:


/tmp/ipykernel_13744/3891104835.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_mode_imputed[col].fillna(mode_value, inplace=True)


,Numerical_Col_A,Numerical_Col_B,Categorical_Col_C,Categorical_Col_D
0,10.0,1.0,A,X
1,20.0,NaN,B,Y
2,NaN,3.0,A,Z
3,40.0,4.0,A,X
4,50.0,5.0,C,X
5,60.0,NaN,B,Y
6,NaN,7.0,A,Z
7,80.0,8.0,C,X
8,90.0,9.0,A,Y
9,100.0,10.0,A,X


### Combined Imputation Example

Let's apply both median imputation for numerical columns and mode imputation for categorical columns to a fresh copy of the original DataFrame.

In [6]:
df_combined_imputed = df.copy()

# Impute numerical columns with median
for col in numerical_cols:
    if df_combined_imputed[col].isnull().any():
        median_value = df_combined_imputed[col].median()
        df_combined_imputed[col].fillna(median_value, inplace=True)
        print(f"Numerical column '{col}' imputed with median: {median_value:.2f}")

# Impute categorical columns with mode
for col in categorical_cols:
    if df_combined_imputed[col].isnull().any():
        mode_value = df_combined_imputed[col].mode()[0]
        df_combined_imputed[col].fillna(mode_value, inplace=True)
        print(f"Categorical column '{col}' imputed with mode: {mode_value}")

print("\nDataFrame after Combined Imputation (Median for numerical, Mode for categorical):")
display(df_combined_imputed)

print("\nCheck for any remaining missing values:")
print(df_combined_imputed.isnull().sum())

Numerical column 'Numerical_Col_A' imputed with median: 55.00
Numerical column 'Numerical_Col_B' imputed with median: 6.00
Categorical column 'Categorical_Col_C' imputed with mode: A
Categorical column 'Categorical_Col_D' imputed with mode: X

DataFrame after Combined Imputation (Median for numerical, Mode for categorical):


/tmp/ipykernel_13744/2585867648.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_combined_imputed[col].fillna(median_value, inplace=True)
/tmp/ipykernel_13744/2585867648.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

,Numerical_Col_A,Numerical_Col_B,Categorical_Col_C,Categorical_Col_D
0,10.0,1.0,A,X
1,20.0,6.0,B,Y
2,55.0,3.0,A,Z
3,40.0,4.0,A,X
4,50.0,5.0,C,X
5,60.0,6.0,B,Y
6,55.0,7.0,A,Z
7,80.0,8.0,C,X
8,90.0,9.0,A,Y
9,100.0,10.0,A,X



Check for any remaining missing values:
Numerical_Col_A      0
Numerical_Col_B      0
Categorical_Col_C    0
Categorical_Col_D    0
dtype: int64
